# Daily accumulation for non-UTC timezone for the ERA5-land accumulated variables

The following workflow demonstrates how to calculate the daily sum of the accumulate variables in ERA5-land data for a non-UTC time-zone.

The crux of this problem is that ERA5-land accumulates data from 00:00-UTC to 00:00-UTC. We must sample the data at the correct time steps, and then sum the relevant components to give an accumulation based on the time-zone of interest.

In this exmaple, we compute the daily accumulation of precipitation for a 10x10 centres on Romania, where we want use a time zone of UTC+02:00.


In [1]:
# Import the libraries used in this notebook
import cdsapi
import xarray as xr
import pandas as pd

# Set the time zone shift as a variable so it is easy to change
TIME_ZONE_SHIFT_HOURS = +2

# Calculate the time the TIME-ZONE midnight in UTC
LOCAL_MIDNIGHT_IN_UTC = (0-TIME_ZONE_SHIFT_HOURS) % 24

## Download some raw hourly data

Here we choose the ERA5 land total_precipitation data for January 2024. As the data is partly accumulated in the archive, we just need to retrieve the time-steps which affect our calculation, i.e. `00:00` UTC and local midnight expressed in UTC (`LOCAL_MIDNIGHT_IN_UTC`).

We have chosen a coarse grid to reduce the size of the download for this demonstration.

In [2]:
TIME_STEPS = ['00:00', f"{LOCAL_MIDNIGHT_IN_UTC:02d}:00"]
client = cdsapi.Client()
dataset = "reanalysis-era5-land"
request = {
    'product_type': ['reanalysis'],
    'variable': ['total_precipitation'],
    'date': '20240101/20240131',
    'time': TIME_STEPS,
    'area': [50, 20, 40, 30],
    'grid': [1, 1],
    'data_format': 'grib',
    'download_format': "unarchived"
}
result_file = client.retrieve(dataset, request).download()


2026-11-16 15:25:19,007 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.
<br>
The [ADS Terms of Use](https://ads.atmosphere.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change]( https://forum.ecmwf.int/t/upcoming-ads-terms-of-use-change/15375)**
2026-11-16 15:25:19,008 INFO [2024-07-18T00:00:00Z] Remember that you need to have an ECMWF account to use CDS-Beta. **Your existing CDS c

33d479fc183df3fe83b482b1fe34f831.grib:   0%|          | 0.00/27.2k [00:00<?, ?B/s]

## Open the result file with xarray

Included here are the additional keyword arguments used in the backend by the CDS.

In [4]:
ds = xr.open_dataset(
    result_file, time_dims=["valid_time"],
    coords_as_attributes=["surface", "number"],
)
print(ds)

Ignoring index file '33d479fc183df3fe83b482b1fe34f831.grib.46b27.idx' older than GRIB file


<xarray.Dataset> Size: 31kB
Dimensions:     (valid_time: 62, latitude: 11, longitude: 11)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 496B 2024-01-01 ... 2024-01-31T22...
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
Data variables:
    tp          (valid_time, latitude, longitude) float32 30kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...


## Compute the daily sum

ERA5-Land accumulations restart every day at `00:00 UTC`:

* the field at `HH:00 UTC` (for `HH` = `01` to `23`) holds the accumulation from `00:00 UTC` on the same day until `HH:00 UTC`;
* the field at `00:00 UTC` holds the accumulation for the whole of the **previous** UTC day.

A local day runs from one local midnight to the next, and exactly one UTC midnight falls between them. The local daily total is therefore the sum of two parts:

1. the accumulation from the local midnight at the **start** of the local day until the next `00:00 UTC`: the `00:00 UTC` field minus the field at the local midnight at the start of the day;
2. the accumulation from that `00:00 UTC` until the local midnight at the **end** of the local day: simply the field at the local midnight at the end of the day.

First we split the data into the UTC-midnight and local-midnight fields.

In [5]:
ds_UTC_midnight = ds.sel(valid_time=ds.valid_time.dt.hour == 0)
ds_local_midnight = ds.sel(valid_time=ds.valid_time.dt.hour == LOCAL_MIDNIGHT_IN_UTC)

print(ds_UTC_midnight)

<xarray.Dataset> Size: 15kB
Dimensions:     (valid_time: 31, latitude: 11, longitude: 11)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
Data variables:
    tp          (valid_time, latitude, longitude) float32 15kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...


To add and subtract the two datasets, we label each field with the **local** date on which it falls, i.e. we convert `valid_time` to local time and keep only the date.

After this relabelling, for a local date `D`:

* `ds_local_midnight` at `D` is the field at the local midnight at the start of `D`;
* `ds_UTC_midnight` at `D` is the field at the UTC midnight that falls within local day `D`.

In [6]:
ds_UTC_midnight = ds_UTC_midnight.assign_coords(
    valid_time=(ds_UTC_midnight.valid_time + pd.Timedelta(hours=TIME_ZONE_SHIFT_HOURS)).dt.floor("D")
)
ds_local_midnight = ds_local_midnight.assign_coords(
    valid_time=(ds_local_midnight.valid_time + pd.Timedelta(hours=TIME_ZONE_SHIFT_HOURS)).dt.floor("D")
)
print(ds_local_midnight)

<xarray.Dataset> Size: 15kB
Dimensions:     (latitude: 11, longitude: 11, valid_time: 31)
Coordinates:
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
  * valid_time  (valid_time) datetime64[ns] 248B 2024-01-02 ... 2024-02-01
Data variables:
    tp          (valid_time, latitude, longitude) float32 15kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...


/var/folders/l2/529q7bzs665bnrn7_wjx1nsr0000gn/T/ipykernel_20924/2504421026.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  valid_time=(ds_UTC_midnight.valid_time + pd.Timedelta(hours=TIME_ZONE_SHIFT_HOURS)).dt.floor("D")
/var/folders/l2/529q7bzs665bnrn7_wjx1nsr0000gn/T/ipykernel_20924/2504421026.py:5: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  valid_time=(ds_local_midnight.valid_time + pd.Timedelta(hours=TIME_ZONE_SHIFT_HOURS)).dt.floor("D")


Part 1 is the accumulation from the local midnight at the start of the day until the next `00:00 UTC`. This is the UTC-midnight field minus the local-midnight field, both labelled with the same local date.

In [7]:
ds_local_to_utc_midnight = ds_UTC_midnight - ds_local_midnight
print(ds_local_to_utc_midnight)

<xarray.Dataset> Size: 15kB
Dimensions:     (latitude: 11, longitude: 11, valid_time: 30)
Coordinates:
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
  * valid_time  (valid_time) datetime64[ns] 240B 2024-01-02 ... 2024-01-31
Data variables:
    tp          (valid_time, latitude, longitude) float32 15kB 2.027e-06 ... ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...


Part 2 is the accumulation from `00:00 UTC` until the local midnight at the end of the day. The local midnight at the end of day `D` is the local midnight at the start of day `D+1`, so we take the local-midnight field and shift its label back by one day.

In [8]:
ds_utc_to_local_midnight = ds_local_midnight.assign_coords(
    valid_time=ds_local_midnight.valid_time - pd.Timedelta(days=1)
)
print(ds_utc_to_local_midnight)

<xarray.Dataset> Size: 15kB
Dimensions:     (latitude: 11, longitude: 11, valid_time: 31)
Coordinates:
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
  * valid_time  (valid_time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
Data variables:
    tp          (valid_time, latitude, longitude) float32 15kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...


/var/folders/l2/529q7bzs665bnrn7_wjx1nsr0000gn/T/ipykernel_20924/2326607129.py:2: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  valid_time=ds_local_midnight.valid_time - pd.Timedelta(days=1)


Finally, we sum the two parts to get the total precipitation for each day in the local time zone. Xarray only keeps the dates present in both datasets, so incomplete days at the start or end of the period are dropped.

The same code works for both positive and negative time zone shifts, so no further adjustment is needed.

In [9]:
ds_accum_local = ds_local_to_utc_midnight + ds_utc_to_local_midnight
print(ds_accum_local)

<xarray.Dataset> Size: 15kB
Dimensions:     (latitude: 11, longitude: 11, valid_time: 30)
Coordinates:
  * latitude    (latitude) float64 88B 50.0 49.0 48.0 47.0 ... 42.0 41.0 40.0
  * longitude   (longitude) float64 88B 20.0 21.0 22.0 23.0 ... 28.0 29.0 30.0
  * valid_time  (valid_time) datetime64[ns] 240B 2024-01-02 ... 2024-01-31
Data variables:
    tp          (valid_time, latitude, longitude) float32 15kB 0.002138 ... 0...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-11-16T15:25 GRIB to CDM+CF via cfgrib-0.9.1...
